# GTFS Scheduled Trips and Stops Analysis

This script processes GTFS data to analyze transit service for a selection of agencies across different days of the week. The main steps are:
1. Filter trips by agency and date: Selects scheduled trips for the specified agencies on a Sunday, Saturday, and a weekday.
2. Extract feed keys and retrieve stops: For each day, unique feed keys from the trips are used to query the corresponding daily scheduled stops, including stop identifiers, names, and associated route information.
3. Expand route-stop relationships: Many stops are served by multiple routes. The script transforms the stops data so that each row represents a unique stop × route combination.
4. Merge with trip counts: Trip-level data is aggregated by feed and route to calculate the number of trips per route. This information is merged with the expanded stops data, providing trip counts for each stop × route combination.
5. Aggregate stop-level statistics: Stops are summarized by feed, stop ID, stop name, and route type to calculate:
- Total number of trips serving each stop.
- Number of distinct routes serving each stop.

The final dataset provides a detailed overview of transit service at the stop level, including how many trips and routes serve each stop for the selected agencies and dates.

In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
import datetime as dt
from calitp_data_analysis.sql import get_engine
from shared_utils import gtfs_utils_v2
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
from pandas.tseries.holiday import USFederalHolidayCalendar
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

In [3]:
# GCS FILE PATH
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [4]:
agencies_to_find = [
    "Bay Area 511 BART Schedule",
    "Big Blue Bus Schedule",
    "Burbank Schedule",
    "Caltrain Schedule",
    "Culver City Schedule",
    "Foothill Schedule",
    "Fresno Schedule",
    "Gold Coast Schedule",
    "Golden Gate Park Shuttle Schedule",
    "Bay Area 511 Golden Gate Transit Schedule",
    "Long Beach Schedule",
    "OCTA Schedule",
    "OmniTrans Schedule",
    "Riverside Schedule",
    "Sacramento Schedule",
    "Santa Cruz Schedule",
    "SBMTD Schedule",
    "SamTrans Schedule",
    "San Diego Schedule"
]

agency_sql = ', '.join(f"'{agency}'" for agency in agencies_to_find)

In [5]:
feed_keys = [
    '7a3f513c343b16a30c135ed7d332b6d6',  # Big Blue Bus Schedule
    'cc6a68a39d22c29b49116584971e69a8',  # Burbank Schedule 
    'f6774d861953d4f4cdcffec95e2652c7',  # Culver City Schedule
    '661ef844bdaa253e8b950740f76061b1',  # Foothill Schedule
    '3cb676436aad669e52042c0e97a9a051',  # Gold Coast Schedule
    'de77cb40e92fb47fa8d16228977cfb86',  # Golden Gate Transit Schedule
    'cddd375786d835389a7beb9632369907',  # Long Beach Schedule
    'cd299184726656597ae2cdb4f4e81e4a',  # OCTA Schedule
    '209e55541803479b23ebb6eea4fae5fa',  # OmniTrans Schedule
    '6eb2b575bee157dace7a2c7155d3cb25',  # Riverside Schedule
    '1eb334a547f5c2cdbdce3f7cc7b03e2b',  # Sacramento Schedule (SacRT)
    'db97cc02836aa5f0cf647d80160b23ec',  # SamTrans Schedule
    '1fff52f9349da228c56eef492df5001b',  # San Diego Schedule (SDMTS)
    '4e549244dd7ce383a3f4337f00134b27',  # BART Schedule
    'f189d5677d4a106b98585f3c5d4fd42c',  # Caltrain Schedule
    '23d1893801eefadf7544a670a3bcd312',  # Fresno Schedule
    'ea33d4691b573336fc9c43c23fa90f65',  # Golden Gate Park Shuttle
    '5a075de618b4d2d2383550863fc8e44e',  # Santa Cruz Schedule
    '6b5c8acdaa4dcb280591578fcbf6c18e'   # SBMTD Schedule
]

In [6]:
feed_keys_str = ", ".join(f"'{k}'" for k in feed_keys)

# Query only those feed_keys
with db_engine.connect() as connection:
    query = f"""
        SELECT 
            feed_key, stop_id, _feed_valid_from, n_hours_in_service, daily_arrivals, arrivals_early_am, arrivals_am_peak, arrivals_midday, 
            arrivals_pm_peak, arrivals_evening, route_id_array, route_type_array, stop_key, tts_stop_name,
            pt_geom, stop_name, location_type, stop_desc, stop_code
        FROM cal-itp-data-infra.mart_gtfs.fct_daily_scheduled_stops
        WHERE service_date = DATE('2025-05-21')
          AND feed_key IN ({feed_keys_str})
    """
    stops_unique_weekday = pd.read_sql(query, connection)

In [7]:
# Query only those feed_keys
with db_engine.connect() as connection:
    query = f"""
        SELECT 
            feed_key, stop_id, _feed_valid_from, n_hours_in_service, daily_arrivals, arrivals_early_am, arrivals_am_peak, arrivals_midday, 
            arrivals_pm_peak, arrivals_evening, route_id_array, route_type_array, stop_key, tts_stop_name,
            pt_geom, stop_name, location_type, stop_desc, stop_code
        FROM cal-itp-data-infra.mart_gtfs.fct_daily_scheduled_stops
        WHERE service_date = DATE('2025-05-17')
          AND feed_key IN ({feed_keys_str})
    """
    stops_unique_saturday = pd.read_sql(query, connection)

In [8]:
# Query only those feed_keys
with db_engine.connect() as connection:
    query = f"""
        SELECT 
            feed_key, stop_id, _feed_valid_from, n_hours_in_service, daily_arrivals, arrivals_early_am, arrivals_am_peak, arrivals_midday, 
            arrivals_pm_peak, arrivals_evening, route_id_array, route_type_array, stop_key, tts_stop_name,
            pt_geom, stop_name, location_type, stop_desc, stop_code
        FROM cal-itp-data-infra.mart_gtfs.fct_daily_scheduled_stops
        WHERE service_date = DATE('2025-05-17')
          AND feed_key IN ({feed_keys_str})
    """
    stops_unique_sunday = pd.read_sql(query, connection)

In [9]:
stops_unique_weekday.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27409 entries, 0 to 27408
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   feed_key            27409 non-null  object             
 1   stop_id             27409 non-null  object             
 2   _feed_valid_from    27409 non-null  datetime64[ns, UTC]
 3   n_hours_in_service  27409 non-null  int64              
 4   daily_arrivals      27409 non-null  int64              
 5   arrivals_early_am   27409 non-null  int64              
 6   arrivals_am_peak    27409 non-null  int64              
 7   arrivals_midday     27409 non-null  int64              
 8   arrivals_pm_peak    27409 non-null  int64              
 9   arrivals_evening    27409 non-null  int64              
 10  route_id_array      27409 non-null  object             
 11  route_type_array    27409 non-null  object             
 12  stop_key            27409 non-nu

In [10]:
# Found that stop_id in BART is different for each platform so combining them before. 

# List of dataframes to process
stops_dfs = [stops_unique_weekday, stops_unique_saturday, stops_unique_sunday]

# Target feed_key to collapse platform stop_ids
target_feed_key = '4e549244dd7ce383a3f4337f00134b27'

# Apply in-place transformation to all dataframes
for df in stops_dfs:
    mask = df['feed_key'] == target_feed_key
    df.loc[mask, 'stop_id'] = df.loc[mask, 'stop_id'].astype(str).str[:-1]

In [11]:
def expand_routes_fixed(row):
    """
    Expands a stop row with multiple route_ids and route_types into separate rows
    for each route, keeping existing pt_geom and stop_code.

    Uses existing daily_arrivals as arrivals_all_day.
    """

    # Ensure arrays exist
    route_ids = row['route_id_array'] if isinstance(row['route_id_array'], list) else []
    route_types = row['route_type_array'] if isinstance(row['route_type_array'], list) else []

    # If only one route_type but multiple route_ids, replicate route_type
    if len(route_types) == 1 and len(route_ids) > 1:
        route_types = route_types * len(route_ids)

    # Pair routes with types
    pairs = list(zip(route_ids, route_types))

    # Pull values from row
    stop_code_value = row.get('stop_code', pd.NA)
    pt_geom_value = row.get('pt_geom', pd.NA)
    daily_arrivals = row.get('daily_arrivals', 0)

    # Handle case with no routes
    if not pairs:
        return pd.DataFrame({
            'feed_key': [row['feed_key']],
            'stop_id': [row['stop_id']],
            'stop_code': [stop_code_value],
            'stop_name': [row['stop_name']],
            'route_id': [pd.NA],
            'route_type': [pd.NA],
            'pt_geom': [pt_geom_value],
            'arrivals_all_day': [daily_arrivals]
        })

    # Expand into multiple rows
    return pd.DataFrame({
        'feed_key': [row['feed_key']] * len(pairs),
        'stop_id': [row['stop_id']] * len(pairs),
        'stop_code': [stop_code_value] * len(pairs),
        'stop_name': [row['stop_name']] * len(pairs),
        'route_id': [r[0] for r in pairs],
        'route_type': [r[1] for r in pairs],
        'pt_geom': [pt_geom_value] * len(pairs),
        'arrivals_all_day': [daily_arrivals] * len(pairs)
    })

In [12]:
stops_expanded_weekday = pd.concat([expand_routes_fixed(r) for _, r in stops_unique_weekday.iterrows()], ignore_index=True)
stops_expanded_saturday = pd.concat([expand_routes_fixed(r) for _, r in stops_unique_saturday.iterrows()], ignore_index=True)
stops_expanded_sunday = pd.concat([expand_routes_fixed(r) for _, r in stops_unique_sunday.iterrows()], ignore_index=True)

In [13]:
def aggregate_stops_listcodes(stops_expanded_df):
    """
    Merge expanded stops with trip counts and aggregate stop × route-level statistics,
    combining multiple stop_codes into a list if they exist.
    Adds n_routes per stop × route group.

    Parameters:
    - stops_expanded_df: pd.DataFrame with stop × route rows (expanded)
    - trips_per_route_df: pd.DataFrame with number of trips per feed_key × route_id × route_type

    Returns:
    - pd.DataFrame aggregated by feed_key, stop_id, stop_name, route_id, route_type
      with total trips per stop × route, stop_codes as lists if multiple exist,
      and n_routes indicating number of unique route_ids per group
    """
    # Ensure route_type and route_id columns have the same type
    stops_expanded_df['route_type'] = stops_expanded_df['route_type'].astype(str)
    stops_expanded_df['route_id'] = stops_expanded_df['route_id'].astype(str)

    # Aggregate per stop × route
    stops_aggregated = stops_expanded_df.groupby(
        ['feed_key', 'stop_id', 'stop_name'],
        dropna=False
    ).agg(
        # n_trips=('n_trips', 'sum'),
        n_arrivals = ('arrivals_all_day', 'first'),
        n_routes=('route_id', 'nunique'),  # number of unique route_ids per group
        stop_code=('stop_code', 'first'),
        pt_geom=('pt_geom', lambda x: next((v for v in x if pd.notna(v)), pd.NA))).reset_index()

    return stops_aggregated

In [14]:
# Aggregate
agg_with_route_weekday = aggregate_stops_listcodes(stops_expanded_weekday)
agg_with_route_saturday = aggregate_stops_listcodes(stops_expanded_saturday)
agg_with_route_sunday = aggregate_stops_listcodes(stops_expanded_sunday)

In [15]:
agg_with_route_weekday.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27355 entries, 0 to 27354
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   feed_key    27355 non-null  object
 1   stop_id     27355 non-null  object
 2   stop_name   27355 non-null  object
 3   n_arrivals  27355 non-null  int64 
 4   n_routes    27355 non-null  int64 
 5   stop_code   24376 non-null  object
 6   pt_geom     27355 non-null  object
dtypes: int64(2), object(5)
memory usage: 1.5+ MB


In [16]:
# Rename columns per day before concatenating
stops_expanded_weekday['day_type'] = 'weekday'
stops_expanded_saturday['day_type'] = 'saturday'
stops_expanded_sunday['day_type'] = 'sunday'

all_days_expanded = pd.concat(
    [stops_expanded_weekday, stops_expanded_saturday, stops_expanded_sunday],
    ignore_index=True
)

agg_per_stop_by_day = all_days_expanded.groupby(
    ['feed_key', 'stop_id', 'stop_name', 'day_type']
).agg(
    total_arrivals=('arrivals_all_day', 'sum'),
    n_routes=('route_id', 'nunique'),
    pt_geom=('pt_geom', lambda x: next((v for v in x if pd.notna(v)), pd.NA))
).reset_index()

# Pivot to wide format so each day is a column
agg_per_stop_allday = agg_per_stop_by_day.pivot(
    index=['feed_key', 'stop_id', 'stop_name', 'pt_geom'],
    columns='day_type',
    values=['total_arrivals', 'n_routes']
).reset_index()

# Flatten multiindex columns
agg_per_stop_allday.columns = ['_'.join(filter(None, col)).strip('_') for col in agg_per_stop_allday.columns]

# Now you can take maximums
agg_per_stop_allday['max_arrivals'] = agg_per_stop_allday[['total_arrivals_weekday', 'total_arrivals_saturday', 'total_arrivals_sunday']].max(axis=1)
agg_per_stop_allday['max_n_routes'] = agg_per_stop_allday[['n_routes_weekday', 'n_routes_saturday', 'n_routes_sunday']].max(axis=1)

# sort
agg_per_stop_allday = agg_per_stop_allday.sort_values('stop_name').reset_index(drop=True)

In [17]:
# Map feed_key directly to organization name
feed_key_to_schedule = {
    '7a3f513c343b16a30c135ed7d332b6d6': 'Big Blue Bus Schedule',
    'cc6a68a39d22c29b49116584971e69a8': 'Burbank Schedule',
    'f6774d861953d4f4cdcffec95e2652c7': 'Culver City Schedule',
    '661ef844bdaa253e8b950740f76061b1': 'Foothill Schedule',
    '3cb676436aad669e52042c0e97a9a051': 'Gold Coast Schedule',
    'de77cb40e92fb47fa8d16228977cfb86': 'Golden Gate Transit Schedule',
    'cddd375786d835389a7beb9632369907': 'Long Beach Schedule',
    'cd299184726656597ae2cdb4f4e81e4a': 'OCTA Schedule',
    '209e55541803479b23ebb6eea4fae5fa': 'OmniTrans Schedule',
    '6eb2b575bee157dace7a2c7155d3cb25': 'Riverside Schedule',
    '1eb334a547f5c2cdbdce3f7cc7b03e2b': 'Sacramento Schedule',
    'db97cc02836aa5f0cf647d80160b23ec': 'SamTrans Schedule',
    '1fff52f9349da228c56eef492df5001b': 'San Diego Schedule',
    '4e549244dd7ce383a3f4337f00134b27': 'BART Schedule',
    'f189d5677d4a106b98585f3c5d4fd42c': 'Caltrain Schedule',
    '23d1893801eefadf7544a670a3bcd312': 'Fresno Schedule',
    'ea33d4691b573336fc9c43c23fa90f65': 'Golden Gate Park Shuttle Schedule',
    '5a075de618b4d2d2383550863fc8e44e': 'Santa Cruz Schedule',
    '6b5c8acdaa4dcb280591578fcbf6c18e': 'SBMTD Schedule'
}



In [18]:
# Add organization_name column
agg_per_stop_allday['name'] = agg_per_stop_allday['feed_key'].map(feed_key_to_schedule)
agg_with_route_weekday['name'] = agg_with_route_weekday['feed_key'].map(feed_key_to_schedule)
agg_with_route_saturday['name'] = agg_with_route_saturday['feed_key'].map(feed_key_to_schedule)
agg_with_route_sunday['name'] = agg_with_route_sunday['feed_key'].map(feed_key_to_schedule)

In [19]:
agg_with_route_weekday= agg_with_route_weekday.sort_values('stop_name').reset_index(drop=True)
agg_with_route_weekday.head(5)

,feed_key,stop_id,stop_name,n_arrivals,n_routes,stop_code,pt_geom,name
0,3cb676436aad669e52042c0e97a9a051,VNACLR:1,.,14,1,None,POINT(-119.294028 34.343645),Gold Coast Schedule
1,661ef844bdaa253e8b950740f76061b1,493,10 Freeway and Azusa Ave E,110,2,493,POINT(-117.908253 34.071995),Foothill Schedule
2,661ef844bdaa253e8b950740f76061b1,494,10 Freeway and Azusa Ave W,113,2,494,POINT(-117.906536 34.072337),Foothill Schedule
3,661ef844bdaa253e8b950740f76061b1,496,10 Freeway and Puente Ave E,66,1,496,POINT(-117.960458 34.070065),Foothill Schedule
4,661ef844bdaa253e8b950740f76061b1,497,10 Freeway and Puente Ave W,70,1,497,POINT(-117.959866 34.070698),Foothill Schedule


In [20]:
agg_with_route_weekday.to_csv(f"{GCS_FILE_PATH}/AHSC_2026/stops_aggregated_weekday.csv", index=False)
agg_with_route_saturday.to_csv(f"{GCS_FILE_PATH}/AHSC_2026/stops_aggregated_saturday.csv", index=False)
agg_with_route_sunday.to_csv(f"{GCS_FILE_PATH}/AHSC_2026/stops_aggregated_sunday.csv", index=False)
agg_per_stop_allday.to_csv(f"{GCS_FILE_PATH}/AHSC_2026/stops_aggregated_allday.csv", index=False)